In [1]:
import pandas as pd
import os

# ==========================================
# 1. Configuration and Paths
# ==========================================
BASE_PATH = '..'
INPUT_DIR = os.path.join(BASE_PATH, 'includes', 'dados')

# Input Filenames
FILE_CONSUMERS = 'Tabela_Consumidores_Itapua_2015_2025_full.csv'
FILE_COORDS = 'Tabela_Coord_Mat.csv'
FILE_TARIFF = 'Tabela_Subcategoria_Tarifaria.csv'

# Output Filename
OUTPUT_FILE = os.path.join(BASE_PATH, 'includes','Tabela_consumidores_Itapua.csv')

def main():
    print("--- Starting Consumer Table Consolidation ---")

    # ==========================================
    # 2. Data Loading
    # ==========================================
    df_cons = pd.read_csv(os.path.join(INPUT_DIR, FILE_CONSUMERS), sep=';')
    df_coord = pd.read_csv(os.path.join(INPUT_DIR, FILE_COORDS), sep=';')
    df_tariff = pd.read_csv(os.path.join(INPUT_DIR, FILE_TARIFF), sep=';')

    # ==========================================
    # 3. Data Cleaning
    # ==========================================
    # Trim strings in categorical columns
    for col in ['NM_LOCALIDADE', 'NM_CATEGORIATARIFARIA', 'NM_SITUACAO_IMOVEL']:
        df_cons[col] = df_cons[col].str.strip()

    # To avoid duplicates in coordinates (ensure 1-to-1)
    df_coord = df_coord.drop_duplicates(subset=['SK_MATRICULA'])

    # ==========================================
    # 4. Merging Logic
    # ==========================================
    
    # Step 1: Join with Coordinates
    print("Step 1: Merging Consumers with geographic coordinates...")
    df_merged = pd.merge(df_cons, df_coord, on='SK_MATRICULA', how='left')
    print(f"Success 1! Row count: {len(df_merged)}")

    # Step 2: Join with Tariff using COMPOSITE KEY
    print("Step 2: Merging with tariff descriptions using composite keys...")
    
    df_merged = pd.merge(
        df_merged, 
        df_tariff[['ID_CATEGORIATARIFARIA', 'ID_SUBCATEGORIATARIFARIA', 'NM_SUBCATEGORIATARIFARIA']], 
        left_on=['ID_CATEGORIA', 'ID_SUBCATEGORIA'], 
        right_on=['ID_CATEGORIATARIFARIA', 'ID_SUBCATEGORIATARIFARIA'], 
        how='left'
    )

    # Remove redundant ID columns from the join
    df_merged.drop(columns=['ID_CATEGORIATARIFARIA', 'ID_SUBCATEGORIATARIFARIA'], inplace=True)
    
    print(f"Success 2! Row count: {len(df_merged)}")

    # ==========================================
    # 5. Final Formatting
    # ==========================================
    print("Step 3: Exporting final table...")

    final_columns = [
        'SK_MATRICULA', 
        'NM_LOCALIDADE', 
        'NM_CATEGORIATARIFARIA', 
        'NM_SITUACAO_IMOVEL', 
        'NN_MORADORES', 
        'ST_PISCINA', 
        'LAT_GEO', 
        'LONG_GEO'
    ]
    
    # Check if all columns exist before copying
    df_final = df_merged[final_columns].copy()

    # ==========================================
    # 6. Exporting
    # ==========================================
    df_final.to_csv(OUTPUT_FILE, index=False, sep=';', encoding='utf-8')
    
    print(f"--- Process Complete ---")
    print(f"Final records: {len(df_final)} (Original: {len(df_cons)})")
    print(f"Location: {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

--- Starting Consumer Table Consolidation ---


Step 1: Merging Consumers with geographic coordinates...


Success 1! Row count: 19630
Step 2: Merging with tariff descriptions using composite keys...
Success 2! Row count: 19630
Step 3: Exporting final table...
--- Process Complete ---
Final records: 19630 (Original: 19630)
Location: ..\includes\Tabela_consumidores_Itapua.csv
